<a href="https://colab.research.google.com/github/asrazia68-svg/movie-recommender-system/blob/main/movie_recommeder_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MOVIE RECOMMENDATION SYSTEM PROJECT**

In [ ]:
import numpy as np
import pandas as pd
movies = pd.read_csv('/content/tmdb_5000_movies.csv')
movies.head()
#genere,id,keywords,title,overview,
movies = movies[["id","title","overview","genres","keywords"]]
movies.head()
#preprocessing
movies.isnull().sum()
movies.dropna(inplace=True)
movies.duplicated().sum()
import ast
#helper function
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):#convert dict into list
        L.append(i['name'])
    return L

#convert into list
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

movies.head()
movies['overview'][0]
movies['overview'].apply(lambda x:x.split())
movies.head()
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['overview'] = movies['overview'].apply(lambda x:x.split())
movies.head()
movies["tags"] = movies['overview'] + movies["genres"] + movies["keywords"]
movies.head()
new_df = movies[['id','title','tags']]
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))
new_df.head()
new_df['tags'][0]
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())
new_df.head()
#text preprocessing (stemming)
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

new_df['tags'] = new_df['tags'].apply(stem)

# Text Vectorization (with TfidfVectorizer)
from sklearn.feature_extraction.text import TfidfVectorizer
cv = TfidfVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
print(vectors.shape)
print(vectors[0])
feature_names = cv.get_feature_names_out()
print(len(feature_names))

# Cosine Similarity
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

# Recommend Function
def recommend(movie):
    try:
        movie_index = new_df[new_df['title'] == movie].index[0]
        distances = similarity[movie_index]
        movies_list_with_scores = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

        print(f"Recommendations for '{movie}':")
        recommended_similarities = []
        for i in movies_list_with_scores:
            print(f"- {new_df.iloc[i[0]].title} (Similarity: {i[1]:.2f})")
            recommended_similarities.append(i[1])

        if recommended_similarities:
            average_similarity = np.mean(recommended_similarities)
            print(f"\nAverage similarity of recommendations: {average_similarity:.2f}")
            return average_similarity
        else:
            print("No recommendations found.")
            return 0.0
    except IndexError:
        print("Movie not found in dataset. Please check spelling!")
        return None

# Test and get accuracy
movie_to_recommend = 'The Princess and the Frog'
accuracy_score = recommend(movie_to_recommend)

if accuracy_score is not None:
    print(f"The 'accuracy' (average similarity) for '{movie_to_recommend}' is: {accuracy_score:.2f}")

/tmp/ipykernel_7134/174317809.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))
/tmp/ipykernel_7134/174317809.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())
/tmp/ipykernel_7134/174317809.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http

(4800, 5000)
[0. 0. 0. ... 0. 0. 0.]
5000
Recommendations for 'The Princess and the Frog':
- Frozen (Similarity: 0.20)
- Bolt (Similarity: 0.19)
- Enchanted (Similarity: 0.19)
- Pocahontas (Similarity: 0.18)
- Princess Mononoke (Similarity: 0.18)

Average similarity of recommendations: 0.19
The 'accuracy' (average similarity) for 'The Princess and the Frog' is: 0.19
